In [1]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import append_row, evaluate, summary, time_inference, time_inference_torch

import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time
import os

In [2]:
# Constants
NUM_CLASSES = 4
RANDOM_STATE = 1082
TRAIN_SIZE = 0.8
REAL_DATA_PATH = "../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx"
BASE_DIR = "../../../Datasets/excel/2dSpoofingDataset/"
RESULTS_CSV = "../../../2D_Results.csv"

# Hyperparameters
INPUT_SIZE = 1
BATCH_SIZE = 64
HIDDEN_SIZE = 64
NUM_EPOCHS = 100
LEARNING_RATE = 0.001

models_list = ["ctgan", "gc", "tvae"]
sample_sizes = [10_000, 100_000, 1_000_000]

In [3]:
# Load real data (for testing)
real_df = pd.read_excel(REAL_DATA_PATH)
if 'Output' not in real_df.columns:
    raise ValueError("Real data does not have 'Output' column.")
X_real = real_df.drop('Output', axis=1)
y_real = real_df['Output']

In [4]:
# Encode labels if needed
if y_real.dtype == object or y_real.dtype.name == 'category':
    le_real = LabelEncoder()
    y_real = le_real.fit_transform(y_real)
else:
    le_real = None

# Pre-scale real data (scaler will be fit on synthetic train)
X_real = X_real.astype('float32')

In [5]:
# Define LSTM Classifier
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, num_layers=2)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [6]:
results = []

In [7]:
# Define a basic LSTM model for classification
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, num_layers=2)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

In [ ]:
for model_name in models_list:
    for n_samples in sample_sizes:
        file_name = f"spoofing_{model_name}_{n_samples}.csv"
        file_path = os.path.join(BASE_DIR, file_name)
        if not os.path.isfile(file_path):
            print(f"File {file_name} not found, skipping.")
            continue

        # Load synthetic dataset
        synth_df = pd.read_csv(file_path)
        if 'Output' not in synth_df.columns:
            print(f"Skipping {file_name}: 'Output' column not found.")
            continue

        X_synth = synth_df.drop('Output', axis=1)
        y_synth = synth_df['Output']

        # Encode synthetic labels if needed
        if y_synth.dtype == object or y_synth.dtype.name == 'category':
            le_synth = LabelEncoder()
            y_synth = le_synth.fit_transform(y_synth)

        # Check feature match
        if list(X_synth.columns) != list(X_real.columns):
            print(f"Feature mismatch between {file_name} and real data, skipping.")
            continue

        # Train/test split on synthetic
        X_train, _, y_train, _ = train_test_split(
            X_synth, y_synth, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y_synth)

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_real_scaled = scaler.transform(X_real)

        # Reshape for LSTM
        SEQUENCE_LENGTH = X_train_scaled.shape[1]
        X_train_lstm = X_train_scaled.reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)
        X_real_lstm = X_real_scaled.reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)

        # Prepare tensors
        X_train_tensor = torch.tensor(X_train_lstm, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
        X_test_tensor = torch.tensor(X_real_lstm, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_real.values, dtype=torch.long)

        # DataLoaders
        train_data = TensorDataset(X_train_tensor, y_train_tensor)
        test_data = TensorDataset(X_test_tensor, y_test_tensor)
        train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

        # Init model, loss, optimizer
        model = LSTMClassifier(INPUT_SIZE, HIDDEN_SIZE, NUM_CLASSES).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        # Training
        start_time = time.time()
        for epoch in range(NUM_EPOCHS):
            model.train()
            running_loss = 0.0
            correct = 0
            total = 0
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * X_batch.size(0)
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == y_batch).sum().item()
                total += y_batch.size(0)
            if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == NUM_EPOCHS - 1:
                epoch_loss = running_loss / len(train_loader.dataset)
                epoch_acc = correct / total
                print(f"{file_name} | Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")

        train_time = time.time() - start_time

        # Evaluation
        model.eval()
        all_preds = []
        all_probs = []
        all_labels = []
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(device)
                outputs = model(X_batch)
                _, preds = torch.max(outputs, 1)
                all_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_batch.numpy())

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
        print(f"{file_name} | Accuracy: {acc:.4f} | F1: {f1:.4f}")

        # Store results
        _row = evaluate(
            all_labels,
            all_preds,
            model_name="LSTM Classifier",
            dataset_name=file_name,
            family="Synthetic-LSTM",
            labels=list(range(4)),
            class_names=["benign", "simple", "intermediate", "sophisticated"],
            y_proba=all_probs,
            y_train=y_train,
            train_time_sec=train_time,
            num_epochs=NUM_EPOCHS,
            parameters=str(model.state_dict().keys()),
            seed=RANDOM_STATE,
            n_features=None,
            n_train=len(y_train),
            test_set="real_full",
            timing=time_inference_torch(model, next(iter(test_loader))[0].to(device)),
        )
        print(summary(_row))
        results.append(_row)


In [ ]:
# Save results to CSV
append_row(results, RESULTS_CSV)
print(f"Results appended to {RESULTS_CSV}")